In [1]:
from faker import Faker
import random
import pandas as pd

fake = Faker()

def generate_users(n: int = 100) -> pd.DataFrame:
    rows = []

    for i in range(1, n + 1):
        # mostly valid records
        row = {
            "id": i,
            "email": fake.email(),
            "age": random.randint(0, 90),
        }

        # inject some bad data (~20%)
        r = random.random()
        if r < 0.05:
            row["email"] = "not-an-email"          # invalid email
        elif r < 0.10:
            row["age"] = -5                        # invalid age
        elif r < 0.15:
            row["id"] = None                       # missing required
        elif r < 0.20:
            row["age"] = 999                       # unrealistic age

        rows.append(row)

    return pd.DataFrame(rows)

if __name__ == "__main__":
    df = generate_users(20000)
    df.to_csv("../data/users.csv", index=False)
    print("users.csv generated")


users.csv generated


In [2]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from faker import Faker


@dataclass(frozen=True)
class UsersJsonlGenerator:
    """
    Generates fake user records and writes them as JSON Lines (JSONL):
    each line is a standalone JSON object (record).
    """
    output_path: str | Path
    seed: int = 42
    locale: str = "en_US"

    def generate(self, n: int = 1000) -> Path:
        if n <= 0:
            raise ValueError("n must be > 0")

        out = Path(self.output_path)
        out.parent.mkdir(parents=True, exist_ok=True)

        fake = Faker(self.locale)
        Faker.seed(self.seed)

        with out.open("w", encoding="utf-8") as f:
            for i in range(1, n + 1):
                record: dict[str, Any] = {
                    "id": i,  # simple incremental id
                    "first_name": fake.first_name(),
                    "last_name": fake.last_name(),
                    "email": fake.email(),
                    "created_at": fake.iso8601(),  # string timestamp
                    "is_active": fake.boolean(chance_of_getting_true=85),
                    "country": fake.country_code(),
                    # Example of a system id that sometimes has leading zeros:
                    "system_x_id": f"{fake.random_int(min=0, max=999999):06d}",
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        return out


if __name__ == "__main__":
    # pip install faker
    path = UsersJsonlGenerator("../data/users.jsonl").generate(n=2000)
    print(f"Wrote: {path}")

Wrote: ../data/users.jsonl
